In [1]:
from random import seed,randint
from numpy import array
from math import ceil,log10,sqrt,log
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,LSTM,TimeDistributed,RepeatVector

In [2]:
def random_sum_pairs(n_exmples,n_numbers,largest):
    x,y=[],[]
    for i in range(n_exmples):
        in_pattern=[randint(1,largest) for _ in range(n_numbers)]
        out_pattern=sum(in_pattern)
        x.append(in_pattern)
        y.append(out_pattern)
    return x,y

In [3]:
x,y=random_sum_pairs(100,4,20)
print(x)

[[7, 7, 18, 13], [12, 2, 12, 15], [13, 7, 2, 15], [12, 1, 18, 7], [10, 15, 1, 6], [10, 18, 12, 17], [2, 6, 20, 12], [16, 7, 9, 13], [19, 8, 1, 15], [13, 18, 13, 16], [19, 7, 6, 13], [12, 10, 1, 19], [3, 1, 20, 3], [4, 7, 10, 16], [17, 6, 15, 3], [16, 4, 7, 2], [6, 5, 12, 16], [11, 7, 1, 20], [3, 12, 14, 6], [7, 8, 9, 2], [9, 1, 6, 1], [7, 7, 20, 18], [7, 8, 5, 18], [12, 14, 12, 17], [12, 6, 5, 4], [17, 11, 18, 10], [19, 10, 12, 15], [16, 14, 13, 13], [7, 16, 3, 3], [3, 11, 12, 1], [8, 14, 9, 11], [14, 17, 13, 17], [20, 6, 2, 14], [7, 19, 10, 13], [6, 8, 16, 13], [13, 19, 19, 10], [20, 7, 3, 14], [17, 11, 18, 13], [7, 14, 2, 12], [16, 20, 17, 20], [14, 20, 16, 14], [9, 14, 14, 1], [13, 7, 12, 2], [7, 1, 6, 17], [3, 6, 1, 10], [8, 6, 10, 9], [8, 8, 5, 9], [3, 6, 19, 19], [14, 9, 20, 16], [19, 7, 16, 16], [12, 15, 12, 2], [17, 16, 10, 19], [17, 14, 11, 1], [10, 19, 14, 7], [1, 12, 4, 8], [5, 9, 5, 1], [1, 10, 5, 12], [19, 7, 19, 20], [9, 6, 5, 2], [9, 1, 2, 15], [2, 14, 4, 9], [2, 12, 17,

In [4]:
print(y)

[45, 41, 37, 38, 32, 57, 40, 45, 43, 60, 45, 42, 27, 37, 41, 29, 39, 39, 35, 26, 17, 52, 38, 55, 27, 56, 56, 56, 29, 27, 42, 61, 42, 49, 43, 61, 44, 59, 35, 73, 64, 38, 34, 31, 20, 33, 30, 47, 59, 58, 41, 62, 43, 50, 25, 20, 28, 65, 22, 27, 29, 42, 25, 54, 25, 53, 61, 42, 45, 44, 39, 54, 35, 23, 50, 56, 39, 39, 30, 38, 27, 31, 45, 48, 17, 35, 53, 37, 46, 21, 59, 38, 48, 31, 59, 39, 34, 34, 31, 37]


In [5]:
ceil(log10(20+1))

2

In [6]:
def pairs_to_string(x,y,n_numbers,largest):
    max_length=n_numbers*ceil(log10(largest+1))+n_numbers-1
    xstr=[]
    for p in x:
        strp='+'.join([str(n) for n in p])
        strp=''.join([' ' for _ in range(max_length-len(strp))])+strp
        xstr.append(strp)
    max_length=ceil(log10(n_numbers*(largest+1)))
    ystr=[]
    for p in y:
        strp=str(p)
        strp=''.join([' ' for _ in range(max_length-len(strp))])+strp
        ystr.append(strp)
    return xstr,ystr


In [7]:
xstr,ystr=pairs_to_string(x,y,4,20)

In [8]:
def integer_encode(x,y,alphabet):
    char_to_int=dict((c,i) for i,c in enumerate(alphabet))
    print(char_to_int)
    Xenc=[]
    for p in x:
        integer_encoded=[char_to_int[char]for char in p]
        Xenc.append(integer_encoded)
    yenc=[]
    for p in y:
        integer_encoded=[char_to_int[char]for char in p]
        yenc.append(integer_encoded)
    return Xenc,yenc

In [9]:
integer_encode(xstr,ystr,['1','2','3','4','5','6','7','8','9','0',' ','+'])

{'1': 0, '2': 1, '3': 2, '4': 3, '5': 4, '6': 5, '7': 6, '8': 7, '9': 8, '0': 9, ' ': 10, '+': 11}


([[10, 10, 6, 11, 6, 11, 0, 7, 11, 0, 2],
  [10, 0, 1, 11, 1, 11, 0, 1, 11, 0, 4],
  [10, 10, 0, 2, 11, 6, 11, 1, 11, 0, 4],
  [10, 10, 0, 1, 11, 0, 11, 0, 7, 11, 6],
  [10, 10, 0, 9, 11, 0, 4, 11, 0, 11, 5],
  [0, 9, 11, 0, 7, 11, 0, 1, 11, 0, 6],
  [10, 10, 1, 11, 5, 11, 1, 9, 11, 0, 1],
  [10, 10, 0, 5, 11, 6, 11, 8, 11, 0, 2],
  [10, 10, 0, 8, 11, 7, 11, 0, 11, 0, 4],
  [0, 2, 11, 0, 7, 11, 0, 2, 11, 0, 5],
  [10, 10, 0, 8, 11, 6, 11, 5, 11, 0, 2],
  [10, 0, 1, 11, 0, 9, 11, 0, 11, 0, 8],
  [10, 10, 10, 2, 11, 0, 11, 1, 9, 11, 2],
  [10, 10, 3, 11, 6, 11, 0, 9, 11, 0, 5],
  [10, 10, 0, 6, 11, 5, 11, 0, 4, 11, 2],
  [10, 10, 10, 0, 5, 11, 3, 11, 6, 11, 1],
  [10, 10, 5, 11, 4, 11, 0, 1, 11, 0, 5],
  [10, 10, 0, 0, 11, 6, 11, 0, 11, 1, 9],
  [10, 10, 2, 11, 0, 1, 11, 0, 3, 11, 5],
  [10, 10, 10, 10, 6, 11, 7, 11, 8, 11, 1],
  [10, 10, 10, 10, 8, 11, 0, 11, 5, 11, 0],
  [10, 10, 6, 11, 6, 11, 1, 9, 11, 0, 7],
  [10, 10, 10, 6, 11, 7, 11, 4, 11, 0, 7],
  [0, 1, 11, 0, 3, 11, 0, 1, 11, 

In [10]:
def one_hot_encode(x,y,max_int):
    xenc=[]
    for p in x:
        pattern=[]
        for index in p:
            vector=[0 for _ in range(max_int)]
            vector[index]=1
            pattern.append(vector)
        xenc.append(pattern)
    yenc=[]
    for p in y:
        pattern=[]
        for index in p:
            vector=[0 for _ in range(max_int)]
            vector[index]=1
            pattern.append(vector)
        yenc.append(pattern)
    return xenc,yenc

In [11]:
import numpy as np
def generate_data(n_samples,n_numbers,largest,alphabet):
    x,y=random_sum_pairs(n_samples,n_numbers,largest)
    x,y=pairs_to_string(x,y,n_numbers,largest)
    x,y=integer_encode(x,y,alphabet)
    x,y=one_hot_encode(x,y,len(alphabet))
    x,y=np.array(x),np.array(y)
    return x,y

In [12]:
from numpy import argmax
import numpy as np
def integer_decode(seq,alphabet):
    int_to_char=dict((i,c)for i,c in enumerate(alphabet))
    strings=[]
    for p in seq:
        string=int_to_char[argmax(p)]
        strings.append(string)
    return ''.join(strings)

In [13]:
seed(1)
n_sample=1000
n_numbers=3
largest=20
alphabets=['1','2','3','4','5','6','7','8','9','0',' ','+']
n_chars=len(alphabets)
n_in_seq_length=n_numbers*ceil(log10(largest+1))+n_numbers-1
n_out_seq_length=ceil(log10(largest+1))


In [14]:
n_out_seq_length

2

In [15]:
n_in_seq_length

8

In [16]:
x,y=generate_data(n_sample,n_numbers,largest,alphabets)

{'1': 0, '2': 1, '3': 2, '4': 3, '5': 4, '6': 5, '7': 6, '8': 7, '9': 8, '0': 9, ' ': 10, '+': 11}


In [17]:
x.shape

(1000, 8, 12)

In [18]:
y.shape

(1000, 2, 12)

In [19]:
x[0]

array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
       [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
       [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
       [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]])

In [20]:
model=Sequential()
model.add(LSTM(128,input_shape=(n_in_seq_length,n_chars)))
model.add(RepeatVector(n_out_seq_length))
model.add(LSTM(64,return_sequences=True))
model.add(TimeDistributed(Dense(n_chars,activation='softmax')))

C:\Users\LOQ\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [21]:
model.compile(loss='categorical_crossentropy',optimizer='adam',metrics=['accuracy'])


In [22]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 128)            │        72,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 2, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 2, 64)          │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 2, 12)          │           780 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 122,380 (478.05 KB)

 Trainable params: 122,380 (478.05 KB)

 Non-trainable params: 0 (0.00 B)

In [23]:
for i in range(200):
    x,y=generate_data(n_sample,n_numbers,largest,alphabets)
    print('epoch: ',i)
    model.fit(x,y,epochs=1,batch_size=50)

{'1': 0, '2': 1, '3': 2, '4': 3, '5': 4, '6': 5, '7': 6, '8': 7, '9': 8, '0': 9, ' ': 10, '+': 11}
epoch:  0
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.2180 - loss: 2.4231
{'1': 0, '2': 1, '3': 2, '4': 3, '5': 4, '6': 5, '7': 6, '8': 7, '9': 8, '0': 9, ' ': 10, '+': 11}
epoch:  1
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.2290 - loss: 2.1645
{'1': 0, '2': 1, '3': 2, '4': 3, '5': 4, '6': 5, '7': 6, '8': 7, '9': 8, '0': 9, ' ': 10, '+': 11}
epoch:  2
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.2095 - loss: 1.9606
{'1': 0, '2': 1, '3': 2, '4': 3, '5': 4, '6': 5, '7': 6, '8': 7, '9': 8, '0': 9, ' ': 10, '+': 11}
epoch:  3
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.2180 - loss: 1.9211
{'1': 0, '2': 1, '3': 2, '4': 3, '5': 4, '6': 5, '7': 6, '8': 7, '9': 8, '0': 9, ' ': 10, '+': 11}
epoch:  4
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.2040 - loss: 1.9071
{'1': 0, '2': 1, '3': 2, '4': 3, '5': 4, '6': 5, '7': 6, '8': 7, '9': 8, '0': 9, ' ': 10, 

In [37]:
x,y=generate_data(n_in_seq_length,n_numbers,largest,alphabets)
result=model.predict(x,batch_size=50)
result

{'1': 0, '2': 1, '3': 2, '4': 3, '5': 4, '6': 5, '7': 6, '8': 7, '9': 8, '0': 9, ' ': 10, '+': 11}
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step


array([[[3.31988093e-04, 9.99649763e-01, 1.80327825e-05, 1.13232531e-07,
         1.28622455e-08, 3.16289217e-10, 4.47280928e-12, 1.62496665e-12,
         4.53380611e-10, 6.81306886e-08, 1.98922934e-10, 1.94165961e-10],
        [7.64676407e-02, 8.66880894e-01, 5.11602163e-02, 1.72090775e-03,
         1.40219854e-04, 4.82650830e-06, 4.32077798e-07, 8.02706040e-07,
         6.08699520e-05, 3.56306694e-03, 3.74579257e-09, 1.20324621e-08]],

       [[5.41492295e-10, 7.03413932e-08, 5.85781061e-04, 9.98435795e-01,
         9.70523455e-04, 7.82691859e-06, 2.30913813e-08, 6.93421265e-10,
         1.17912347e-10, 3.51269014e-12, 5.18522465e-12, 3.16131876e-09],
        [4.81113791e-03, 6.00433834e-02, 8.20861578e-01, 1.06548220e-01,
         6.38131192e-03, 1.86459554e-04, 1.27376288e-05, 3.81995869e-06,
         6.13869342e-05, 1.08641898e-03, 2.86505951e-07, 3.26112058e-06]],

       [[8.18487297e-06, 9.99969125e-01, 2.25848999e-05, 9.14795990e-08,
         2.77088286e-09, 4.69405521e-11, 4.

In [38]:
integer_decode([result[0]],alphabets)

'2'

In [45]:
expected=[integer_decode(y1,alphabets) for y1 in y]
predicted=[integer_decode(y1,alphabets) for y1 in result]



In [46]:
expected

['22', '42', '24', '23', '39', '36', '33', '33']

In [47]:
predicted

['22', '43', '24', '23', '49', '37', '33', '33']

In [48]:
list(zip(expected,predicted))

[('22', '22'),
 ('42', '43'),
 ('24', '24'),
 ('23', '23'),
 ('39', '49'),
 ('36', '37'),
 ('33', '33'),
 ('33', '33')]

In [51]:
abs(np.array(expected,dtype='int')-np.array(predicted,dtype='int')).sum()/len(expected)

np.float64(1.5)